# 03 — Prétraitement des données

## Projet : Smart City Energy Forecasting — Tetouan

**Correspondance méthodologique :** Ce notebook couvre l'étape **10 (Prétraitement des données)** de la méthodologie KDD.


## Importation des bibliothèques et configuration

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import warnings
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

# Résolution du chemin projet
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

# Importation des modules locaux
from src.data_loader import load_tetouan_data, normalize_raw_column_names
from src.preprocessing import (
    check_10min_measurements_per_hour,
    resample_hourly,
    find_missing_hours,
    handle_missing_values,
    detect_rolling_outliers,
    annotate_load_outliers,
    temporal_train_val_test_split,
    get_base_feature_columns,
    scale_train_val_test,
    save_preprocessing_outputs,
    save_scalers
)

print("✅ Bibliothèques et modules locaux importés avec succès.")


✅ Bibliothèques et modules locaux importés avec succès.


## 10.1 Chargement CSV

**Pourquoi :** Charger les données brutes.
**Interprétation :** Le fichier est chargé avec `pandas.read_csv`. Les noms de colonnes sont ensuite standardisés pour éviter les erreurs dues aux espaces.


In [8]:
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Tetuan City power consumption.csv"

# Chargement manuel pour démonstration
df_raw = pd.read_csv(DATA_PATH)
print("Colonnes originales :", df_raw.columns.tolist())

# Standardisation des noms via le module data_loader
df_raw.columns = normalize_raw_column_names(df_raw.columns)
from src.data_loader import COLUMN_MAPPING
df_raw = df_raw.rename(columns=COLUMN_MAPPING)
print("Colonnes standardisées et renommées :", df_raw.columns.tolist())


Colonnes originales : ['DateTime', 'Temperature', 'Humidity', 'Wind Speed', 'general diffuse flows', 'diffuse flows', 'Zone 1 Power Consumption', 'Zone 2  Power Consumption', 'Zone 3  Power Consumption']
Colonnes standardisées et renommées : ['datetime', 'temperature', 'humidity', 'wind_speed', 'general_diffuse_flows', 'diffuse_flows', 'zone1_power', 'zone2_power', 'zone3_power']


## 10.2 Conversion `DateTime`

**Pourquoi :** Permettre les opérations temporelles.
**Interprétation :** La colonne `datetime` est convertie au format `datetime64`. Toute valeur non convertible est identifiée.

In [9]:
df_raw['datetime'] = pd.to_datetime(df_raw['datetime'], errors='coerce')

# Vérification des erreurs de conversion
n_errors = df_raw['datetime'].isna().sum()
print(f"Valeurs non convertibles en datetime : {n_errors}")


Valeurs non convertibles en datetime : 0


## 10.3 Tri chronologique

**Pourquoi :** Le tri garantit que les lags et splits temporels sont corrects.

In [10]:
df_raw = df_raw.sort_values('datetime')
print("✅ Tri chronologique effectué.")


✅ Tri chronologique effectué.


## 10.4 Indexation temporelle

**Pourquoi :** Permet d'utiliser `resample`, `rolling`, `shift` et les features calendaires.

In [11]:
df_raw = df_raw.set_index('datetime')
print(f"Index : {df_raw.index.name} ({df_raw.index.dtype})")


Index : datetime (datetime64[ns])


## 10.5 Vérification de la fréquence 10 minutes

**Pourquoi :** Garantir que les mesures sont régulières.
**Interprétation :** La fréquence doit être inférée ou reconstruite. Dans le fichier fourni, elle est régulière à 10 minutes.

In [12]:
freq = pd.infer_freq(df_raw.index)
print("Fréquence inférée :", freq)

# Vérification robuste via le module preprocessing
hourly_counts = check_10min_measurements_per_hour(df_raw, expected_per_hour=6, raise_error=False)
print("Distribution du nombre d'observations par heure :")
display(hourly_counts.value_counts())


Fréquence inférée : 10min
Distribution du nombre d'observations par heure :


6    8736
Name: count, dtype: int64

## 10.6 Vérification des valeurs manquantes

**Pourquoi :** Identifier les trous dans les mesures.
**Interprétation :** Même si aucune valeur n'est détectée dans le fichier fourni, le code doit pouvoir gérer des valeurs absentes.

In [13]:
print("Valeurs manquantes par colonne :")
display(df_raw.isna().sum())

# Si des valeurs étaient manquantes, on utiliserait handle_missing_values du preprocessing
df_clean = handle_missing_values(df_raw)


Valeurs manquantes par colonne :


temperature              0
humidity                 0
wind_speed               0
general_diffuse_flows    0
diffuse_flows            0
zone1_power              0
zone2_power              0
zone3_power              0
dtype: int64

## 10.7 Traitement des doublons

**Pourquoi :** Assurer l'unicité de chaque timestamp.
**Interprétation :** Les doublons temporels sont supprimés en conservant la première occurrence.

In [14]:
n_duplicates = df_clean.index.duplicated().sum()
print(f"Nombre de doublons temporels : {n_duplicates}")

if n_duplicates > 0:
    df_clean = df_clean[~df_clean.index.duplicated(keep='first')]
    print("✅ Doublons supprimés.")


Nombre de doublons temporels : 0


## 10.10 Création de `target` et `total_load`

**Pourquoi :** Faciliter l'accès aux cibles métier.
**Remarque :** Ces variables sont calculées avant le resampling.

In [15]:
df_clean["target"] = df_clean["zone1_power"]
df_clean["total_load"] = df_clean["zone1_power"] + df_clean["zone2_power"] + df_clean["zone3_power"]
print("✅ Variables `target` et `total_load` créées.")


✅ Variables `target` et `total_load` créées.


## 10.8 Traitement des outliers

**Pourquoi :** Détecter et gérer les points extrêmes.
**Interprétation :** Les outliers sont détectés par Z-score glissant. Les pics réels (métier) sont conservés et annotés pour les modèles de robustesse.

In [16]:
# Utilisation de la fonction dédiée qui calcule le z-score glissant
df_clean = annotate_load_outliers(
    df_clean,
    target_col="target",
    total_load_col="total_load",
    window=24,
    threshold=3.5
)

print("Nombre d'anomalies détectées sur la target :", df_clean['is_target_outlier'].sum())


Nombre d'anomalies détectées sur la target : 1885


## 10.9 Resampling horaire

**Pourquoi :** Réduit le bruit et rend les modèles plus rapides.
**Interprétation :** Agrégation par moyenne sur une fréquence de 1 heure.

In [17]:
df_hourly = resample_hourly(df_clean, rule="1h", check_counts=False)

print(f"Dimensions avant resampling : {df_clean.shape}")
print(f"Dimensions après resampling horaire : {df_hourly.shape}")
print("Fréquence du nouvel index :", pd.infer_freq(df_hourly.index))


Dimensions avant resampling : (52416, 15)
Dimensions après resampling horaire : (8736, 15)
Fréquence du nouvel index : h


## 10.11 Split train / validation / test chronologique

**Pourquoi :** Préparer l'évaluation des modèles sans triche.
**Interprétation :** Le split respecte l'ordre temporel (70% train, 15% validation, 15% test). Aucun shuffle n'est utilisé.

In [18]:
train_df, val_df, test_df = temporal_train_val_test_split(
    df_hourly,
    train_size=0.70,
    val_size=0.15
)

print(f"Ensemble d'entraînement : {train_df.shape[0]} lignes ({train_df.index.min().date()} au {train_df.index.max().date()})")
print(f"Ensemble de validation :  {val_df.shape[0]} lignes ({val_df.index.min().date()} au {val_df.index.max().date()})")
print(f"Ensemble de test :        {test_df.shape[0]} lignes ({test_df.index.min().date()} au {test_df.index.max().date()})")


Ensemble d'entraînement : 6115 lignes (2017-01-01 au 2017-09-12)
Ensemble de validation :  1310 lignes (2017-09-12 au 2017-11-06)
Ensemble de test :        1311 lignes (2017-11-06 au 2017-12-30)


## 10.12 Normalisation sans data leakage

**Pourquoi :** Mettre les variables à la même échelle pour le ML.
**Interprétation :** Les scalers sont ajustés **uniquement sur le train**, puis appliqués à validation et test. Cette règle est essentielle pour éviter la fuite d'information du futur.

In [19]:
# Sélection des colonnes météo autorisées pour le scaling
feature_cols = get_base_feature_columns(df_hourly)
print("Colonnes à normaliser :", feature_cols)

# Normalisation Standard (Z-score)
scaled_outputs = scale_train_val_test(
    train_df,
    val_df,
    test_df,
    feature_cols=feature_cols,
    target_col="target",
    scaler_type="standard"
)

X_train_scaled = scaled_outputs["X_train"]
print("\nAperçu du X_train normalisé :")
display(X_train_scaled.head(3))


Colonnes à normaliser : ['temperature', 'humidity', 'wind_speed', 'general_diffuse_flows', 'diffuse_flows']

Aperçu du X_train normalisé :


,temperature,humidity,wind_speed,general_diffuse_flows,diffuse_flows
datetime,,,,,
2017-01-01 00:00:00,-2.112482,0.462947,-0.815577,-0.735586,-0.661449
2017-01-01 01:00:00,-2.217439,0.620177,-0.815506,-0.735609,-0.661344
2017-01-01 02:00:00,-2.297534,0.704519,-0.815364,-0.735588,-0.661217


## Sauvegarde des datasets préparés et des scalers

Sauvegarde des résultats pour l'étape suivante de modélisation (Feature Engineering et ML).

In [20]:
PREP_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models" / "preprocessing"

# 1. Sauvegarde des dataframes
save_paths = save_preprocessing_outputs(
    output_dir=PREP_DIR,
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    clean_df=df_hourly,
    scaled_outputs=scaled_outputs
)

# 2. Sauvegarde des scalers (pour appliquer en production)
scaler_paths = save_scalers(
    scaler_X=scaled_outputs["scaler_X"],
    scaler_y=scaled_outputs["scaler_y"],
    models_dir=MODEL_DIR,
    prefix="base"
)

print("✅ Sauvegarde terminée :")
print(f"- Datasets propres et splittés dans : {PREP_DIR}")
print(f"- Scalers ajustés dans : {MODEL_DIR}")


✅ Sauvegarde terminée :
- Datasets propres et splittés dans : C:\Users\jarro\OneDrive\Desktop\smart-city-energy-forecasting-tetouan\data\processed
- Scalers ajustés dans : C:\Users\jarro\OneDrive\Desktop\smart-city-energy-forecasting-tetouan\models\preprocessing
